# 01. Setup and Corpus Preparation

**Goal**: end this notebook with a populated Qdrant collection (`eval_corpus`)
and a known list of source documents we can author queries against.

Steps:
1. Verify Qdrant is reachable.
2. Confirm the corpus files are present under `data/corpus/`.
3. Run a programmatic ingest (mirrors `agent-kms ingest --reset`).
4. Print corpus statistics (chunks / docs / heading distribution).

The corpus itself is **already committed** to `data/corpus/`. It consists of
15 markdown files drawn from two public CC-BY-licensed sources — see
`data/corpus/LICENSES.md` for attribution.


In [ ]:
import os
import sys
from pathlib import Path

# Resolve project root so the notebook works whether the CWD is the
# notebooks/ subdir (default for Jupyter) or the project root.
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

# Isolate from personal agent-kms data: dedicated Qdrant instance (port 6433)
# AND a dedicated collection name. The personal Qdrant on :6333 is untouched.
#
# EVAL_MODE selects between the committed public corpus (data/corpus/) and a
# gitignored private corpus (data/private/corpus/). See README "Private
# evaluation mode" for the design — same notebook, two corpora.
sys.path.insert(0, str(ROOT))
from src.eval_utils import resolve_eval_paths

CORPUS_DIR, QUERIES_PATH, COLLECTION = resolve_eval_paths(ROOT)
os.environ["QDRANT_URL"] = "http://localhost:6433"
os.environ["AGENT_KMS_COLLECTION"] = COLLECTION
os.environ["AGENT_KMS_CONFIG"] = str(ROOT / ".agent-kms" / "kms.toml")
os.environ["AGENT_KMS_KNOWLEDGE_ROOT"] = str(CORPUS_DIR)

print("project root:", ROOT)
print("eval mode:   ", os.environ.get("EVAL_MODE", "public"))
print("qdrant:      ", os.environ["QDRANT_URL"])
print("collection:  ", COLLECTION)
print("corpus:      ", CORPUS_DIR)


## 1. Qdrant reachability

In [ ]:
from agent_kms.store import get_client

client = get_client()
existing = [c.name for c in client.get_collections().collections]
print("collections on this Qdrant:", existing)


## 2. Corpus inventory

The corpus is resolved via `EVAL_MODE`: public (default) reads from
`data/corpus/` (committed CC-BY docs), private reads from
`data/private/corpus/` (gitignored). Each file is markdown with
H2-sectioned content suitable for the `markdown_h2` chunker.


In [ ]:
corpus_dir = CORPUS_DIR
docs = sorted(p for p in corpus_dir.glob("*.md") if p.name != "LICENSES.md")
print(f"corpus docs: {len(docs)}")
for d in docs:
    size_kb = d.stat().st_size / 1024
    print(f"  {d.name:60s}  {size_kb:6.1f} KB")


## 3. Ingest into Qdrant

We bypass the CLI's argparse and call `collect_chunks` / `upsert_chunks`
directly so the notebook stays self-contained. `--reset` semantics are
achieved by `ensure_collection(reset=True)`.

Note: the embedding model (`intfloat/multilingual-e5-base`) is fetched
from HuggingFace on first use — the first run downloads ~440 MB.


In [ ]:
from agent_kms.config import load_config, reset_cache
from agent_kms.ingest import collect_chunks, upsert_chunks
from agent_kms.store import ensure_collection

reset_cache()
cfg = load_config()
print("preset:        ", cfg.preset)
print("knowledge_root:", cfg.knowledge_root)
print("collection:    ", cfg.collection)
print("sources:       ", [s.name for s in cfg.sources])

ensure_collection(reset=True)
chunks = collect_chunks(cfg)
print(f"\ncollected {len(chunks)} chunks")

n = upsert_chunks(cfg, chunks)
print(f"upserted  {n} points into '{cfg.collection}'")


## 4. Corpus statistics

In [ ]:
import pandas as pd

rows = []
for c in chunks:
    rows.append({
        "source": Path(c.source_file).name,
        "heading": c.heading,
        "chars": len(c.body),
    })
df_corpus = pd.DataFrame(rows)

print(f"chunks: {len(df_corpus)}")
print(f"docs:   {df_corpus['source'].nunique()}")
print(f"\nchunks per doc:")
print(df_corpus.groupby("source").size().sort_values(ascending=False).to_string())
print(f"\nbody-size summary (chars): min={df_corpus['chars'].min()}, median={int(df_corpus['chars'].median())}, max={df_corpus['chars'].max()}")


## Findings

- **Corpus size**: 15 Japanese technical-doc markdown files → ~130 H2 chunks.
- **Sources**: React 日本語ドキュメント (7 files, CC-BY-4.0) +
  JavaScript Primer 日本語版 (8 files, CC-BY-4.0).
- **Heading distribution is uneven** — the largest files (`async`, `string`,
  `array`, `lifecycle-of-reactive-effects`) contribute 13-15 chunks each;
  smaller pages like `conditional-rendering` contribute only 3.
  This is expected for the `markdown_h2` chunker on real-world docs.
- **Median chunk size ≈ a few hundred characters**; well above the
  `min_chars=80` floor.

This collection is the input to every subsequent notebook.
